# Fine-tuning Qwen2.5-VL for Engagement Prediction

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
TRAIN_JSON_PATH = "../../data/binned/baby_test.json"  # Your original JSON file
TEST_JSON_PATH = "../../data/binned/baby_val.json"    # Your original JSON file
VAL_JSON_PATH = "../../data/binned/baby_val.json"     # Your original JSON file

IMAGE_BASE_DIR = "D:/pixelrec_imgs/cover"  # Base directory for images
OUTPUT_TRAIN_JSON = "train_qwenvl.json"
OUTPUT_VAL_JSON = "val_qwenvl.json"

RANDOM_STATE = 42

## Helper Functions

In [ ]:
def check_image(image_path):
    """
    Check if image file exists and can be opened.
    
    Args:
        image_path: Path to image file
        
    Returns:
        bool: Whether image is valid
    """
    try:
        if not os.path.exists(image_path):
            print(f"Image file not found: {image_path}")
            return False
            
        # Try to open the image
        img = Image.open(image_path)
        img.verify()  # Verify it's a valid image
        return True
    except Exception as e:
        print(f"Error with image {image_path}: {str(e)}")
        return False


def create_conversation(title, tag, description, engagement_label):
    """
    Create conversation format for Qwen2.5-VL.
    
    This format follows the instruction-response pattern where:
    - The model receives an image and text description
    - It needs to predict the engagement level
    """
    # Handle None or missing values
    title = "None" if pd.isna(title) or not title else title
    tag = "None" if pd.isna(tag) or not tag else tag
    description = "None" if pd.isna(description) or not description else description
    
    return [
        {
            "from": "human",
            "value": (
                "<image>\n"
                "Predict the engagement level for this social media post. "
                "Engagement level represents how users interact with the content "
                "(likes, comments, shares, views, favorites). "
                f"The title is: {title}. "
                f"The tag/category is: {tag}. "
                f"The description is: {description}. "
                "What is the engagement level? Choose from: Very Low, Low, Average, High, or Very High."
            )
        },
        {
            "from": "gpt",
            "value": engagement_label
        }
    ]


def convert_to_qwen_format(item, image_base_dir):
    """
    Convert a single data item to Qwen2.5-VL format.
    
    Args:
        item: Dictionary containing the data sample
        image_base_dir: Base directory where images are stored
        
    Returns:
        Dictionary in Qwen format or None if image is invalid
    """
    # Construct full image path
    image_rel_path = item['image']
    # Remove leading '../' or './' if present
    image_rel_path = image_rel_path.replace('../', '').replace('./', '')
    image_full_path = os.path.join(image_base_dir, image_rel_path)
    
    # Check if image is valid
    if not check_image(image_full_path):
        return None
    
    # Extract engagement label and capitalize appropriately
    engagement_label = item['engagement_label']
    # Ensure proper capitalization (e.g., "Very Low" instead of "very low")
    engagement_label = ' '.join(word.capitalize() for word in engagement_label.split())
    
    return {
        "image": image_full_path,
        "conversations": create_conversation(
            item['title'],
            item['tag'],
            item['description'],
            engagement_label
        )
    }

In [ ]:

with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    train_raw_data = json.load(f)
with open(VAL_JSON_PATH, 'r', encoding='utf-8') as f:
    val_raw_data = json.load(f)
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_raw_data = json.load(f)


train_processed_data = []
val_processed_data = []
test_processed_data = []
invalid_count = 0

print("Processing dataset...")
for item in tqdm(train_raw_data):
    qwen_entry = convert_to_qwen_format(item, IMAGE_BASE_DIR)
    if qwen_entry is not None:
        train_processed_data.append(qwen_entry)
    else:
        invalid_count += 1

for item in tqdm(val_raw_data):
    qwen_entry = convert_to_qwen_format(item, IMAGE_BASE_DIR)
    if qwen_entry is not None:
        val_processed_data.append(qwen_entry)
    else:
        invalid_count += 1

for item in tqdm(test_raw_data):
    qwen_entry = convert_to_qwen_format(item, IMAGE_BASE_DIR)
    if qwen_entry is not None:
        test_processed_data.append(qwen_entry)
    else:
        invalid_count += 1

print(f"\nProcessing complete!")

train_data = train_processed_data
val_data = val_processed_data
test_data = test_processed_data

In [ ]:
# label distributions
from collections import Counter

train_labels = [entry['conversations'][1]['value'] for entry in train_data]
val_labels = [entry['conversations'][1]['value'] for entry in val_data]

train_counts = Counter(train_labels)
val_counts = Counter(val_labels)

print("\nTrain set label distribution:")
for label, count in sorted(train_counts.items()):
    percentage = (count / len(train_data)) * 100
    print(f"  {label}: {count} ({percentage:.2f}%)")

print("\nValidation set label distribution:")
for label, count in sorted(val_counts.items()):
    percentage = (count / len(val_data)) * 100
    print(f"  {label}: {count} ({percentage:.2f}%)")

In [ ]:
# # Save training set
# with open(OUTPUT_TRAIN_JSON, 'w', encoding='utf-8') as f:
#     json.dump(train_data, f, ensure_ascii=False, indent=2)

# print(f"Training dataset saved to: {OUTPUT_TRAIN_JSON}")

# # Save validation set
# with open(OUTPUT_VAL_JSON, 'w', encoding='utf-8') as f:
#     json.dump(val_data, f, ensure_ascii=False, indent=2)

# print(f"Validation dataset saved to: {OUTPUT_VAL_JSON}")



In [ ]:
import os
import json
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
from PIL import Image
from dataclasses import dataclass
from typing import Any, Dict, List
import numpy as np
from tqdm import tqdm

from vision_process import process_vision_info

c:\Users\rohai\anaconda3\envs\prac\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [ ]:
# Paths
BASE_MODEL_PATH = "Qwen/Qwen2.5-VL-7B-Instruct"  
# BASE_MODEL_PATH = "Qwen/Qwen2.5-VL-0.5B-Instruct"
TRAIN_JSON_PATH = "train_qwenvl.json"
VAL_JSON_PATH = "val_qwenvl.json"
OUTPUT_DIR = "./qwen_engagement_model"

# Training hyperparameters #TODO: Modify for actual training
BATCH_SIZE = 1  # Adjust based on your GPU memory
GRADIENT_ACCUMULATION_STEPS = 1  # Effective batch size = 16
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WARMUP_STEPS = 1
SAVE_STEPS = 5
EVAL_STEPS = 5
LOGGING_STEPS = 1
MAX_LENGTH = 512

# Set CUDA device
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Change as needed

## Custom Dataset Class

In [ ]:
class EngagementDataset(Dataset):
    def __init__(self, json_path, processor, max_length=512):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.processor = processor
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        conversations = item['conversations']
        image_path = item['image']
        
        image = Image.open(image_path).convert('RGB')
        
        user_text = conversations[0]['value'].replace('<image>\n', '').replace('<image>', '')
        assistant_text = conversations[1]['value']
        
        max_text_tokens = self.max_length - 800  
        
        text_only = f"User: {user_text}\nAssistant: {assistant_text}"
        text_tokens = self.processor.tokenizer(text_only, add_special_tokens=False)['input_ids']
        
        if len(text_tokens) > max_text_tokens:
            # Truncate response
            user_tokens = self.processor.tokenizer(f"User: {user_text}\nAssistant: ", add_special_tokens=False)['input_ids']
            remaining_tokens = max_text_tokens - len(user_tokens)
            assistant_tokens = self.processor.tokenizer(assistant_text, add_special_tokens=False)['input_ids']
            assistant_tokens = assistant_tokens[:remaining_tokens]
            assistant_text = self.processor.tokenizer.decode(assistant_tokens, skip_special_tokens=True)
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": user_text}
                ]
            },
            {
                "role": "assistant",
                "content": assistant_text
            }
        ]
        
        text = self.processor.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt"
        )
        
        # Flatten batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        
        # Set labels
        inputs['labels'] = inputs['input_ids'].clone()
        
        return inputs

## Data Collator

In [ ]:
@dataclass
class DataCollatorForQwen:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:

        input_ids = [f['input_ids'] for f in features]
        attention_masks = [f['attention_mask'] for f in features]
        labels = [f['labels'] for f in features]
        
        # Pad sequences
        max_length = max(len(ids) for ids in input_ids)
        pad_token_id = self.processor.tokenizer.pad_token_id
        
        padded_input_ids = []
        padded_attention_masks = []
        padded_labels = []
        
        for ids, mask, label in zip(input_ids, attention_masks, labels):
            padding_length = max_length - len(ids)
            padded_input_ids.append(torch.cat([ids, torch.full((padding_length,), pad_token_id)]))
            padded_attention_masks.append(torch.cat([mask, torch.zeros(padding_length)]))
            padded_labels.append(torch.cat([label, torch.full((padding_length,), -100)]))  # -100 is ignored in loss
        
        batch = {
            'input_ids': torch.stack(padded_input_ids),
            'attention_mask': torch.stack(padded_attention_masks),
            'labels': torch.stack(padded_labels)
        }
        
        for key in features[0].keys():
            if key not in ['input_ids', 'attention_mask', 'labels']:
                batch[key] = torch.stack([f[key] for f in features])
        
        return batch

## Load Model and Processor

In [ ]:
print("Loading model and processor...")

processor = AutoProcessor.from_pretrained(BASE_MODEL_PATH)

# Load model
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,  # Use bfloat16 for better memory efficiency
    device_map="auto",
    # attn_implementation="flash_attention_2"  # Use Flash Attention if available
    attn_implementation="eager"
)

print(f"Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

Loading model and processor...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.55s/it]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


Model loaded successfully!
Model parameters: 3.75B


## Prepare Datasets

In [ ]:
print("Loading datasets...")

train_dataset = EngagementDataset(
    TRAIN_JSON_PATH, 
    processor, 
    max_length=MAX_LENGTH
)

val_dataset = EngagementDataset(
    VAL_JSON_PATH, 
    processor, 
    max_length=MAX_LENGTH
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

data_collator = DataCollatorForQwen(processor=processor)

Loading datasets...
Train dataset size: 42
Validation dataset size: 8


## Define Training Arguments

In [8]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,  # Use bfloat16 precision
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="tensorboard",
    logging_dir=f"{OUTPUT_DIR}/logs",
    gradient_checkpointing=True,  # Save memory
    optim="adamw_torch",
    lr_scheduler_type="cosine",
)

print("Training arguments configured.")

Training arguments configured.


## Initialize Trainer

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Trainer initialized successfully!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer initialized successfully!


## Start Training

In [10]:
print("\n" + "="*50)
print("Starting fine-tuning...")
print("="*50 + "\n")

# Start training
trainer.train()

print("\n" + "="*50)
print("Training completed!")
print("="*50)


Starting fine-tuning...



`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


: 

## Save Final Model

In [ ]:

final_model_path = os.path.join(OUTPUT_DIR, "final_model")
trainer.save_model(final_model_path)
processor.save_pretrained(final_model_path)

print(f"\n Final model saved to: {final_model_path}")

## Evaluate on Validation Set

In [ ]:
eval_results = trainer.evaluate()

for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

## Test Inference on Sample

In [ ]:
model.eval()
sample_item = val_dataset.data[0]

# Prepare input
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": f"file://{sample_item['image']}"
            },
            {
                "type": "text",
                "text": sample_item['conversations'][0]['value'].replace('<image>\n', '')
            }
        ]
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model.device)


with torch.no_grad():
    generated_ids = model.generate(
        **inputs, 
        max_new_tokens=128, 
        do_sample=False
    )
    generated_ids_trimmed = [
        out_ids[len(in_ids):] 
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )

print(f"\nGround Truth: {sample_item['conversations'][1]['value']}")
print(f"Prediction: {output_text[0]}")